In [ ]:
import cv2
import os
import numpy as np
from PIL import Image
from skimage.feature import local_binary_pattern
from sklearn.svm import SVC
import joblib 

# --- 1. CONFIGURATION ---
# Both models must use the same grid settings for a fair comparison
RADIUS = 1
NEIGHBORS = 8
GRID_X = 8
GRID_Y = 8

# --- 2. HELPER FUNCTIONS ---
def rotate_image(image, angle):
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(image, M, (w, h))
    return rotated

class LocalBinaryPatterns:
    def __init__(self, numPoints, radius, grid_x, grid_y):
        self.numPoints = numPoints
        self.radius = radius
        self.grid_x = grid_x
        self.grid_y = grid_y

    def describe(self, image):
        # Calculates the Spatial Histogram (SVM Feature Vector)
        lbp = local_binary_pattern(image, self.numPoints, self.radius, method="uniform")
        (h, w) = image.shape
        dy = int(h / self.grid_y)
        dx = int(w / self.grid_x)
        histograms = []
        for y in range(0, h - dy + 1, dy):
            for x in range(0, w - dx + 1, dx):
                cell = lbp[y:y+dy, x:x+dx]
                hist, _ = np.histogram(cell.ravel(), bins=np.arange(0, self.numPoints + 3), range=(0, self.numPoints + 2))
                hist = hist.astype("float")
                hist /= (hist.sum() + 1e-7)
                histograms.append(hist)
        return np.hstack(histograms)

# --- 3. DATA LOADING & AUGMENTATION ---
def load_and_augment_data(path):
    image_paths = [os.path.join(path, f) for f in os.listdir(path)]
    
    # Lists for LBPH (Raw Images)
    lbph_faces = []
    lbph_ids = []
    
    # Lists for SVM (Histograms)
    svm_features = []
    svm_labels = []
    
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    desc = LocalBinaryPatterns(NEIGHBORS, RADIUS, GRID_X, GRID_Y)

    print(f"Processing {len(image_paths)} source images...")
    
    for img_path in image_paths:
        try:
            img = Image.open(img_path).convert('L') 
            img_np = np.array(img, 'uint8')
            user_id = int(os.path.split(img_path)[-1].split(".")[1])
            
            # Base Preprocessing
            face_resized = cv2.resize(img_np, (200, 200), interpolation=cv2.INTER_CUBIC)
            face_smooth = cv2.bilateralFilter(face_resized, 5, 75, 75)
            enhanced = clahe.apply(face_smooth)
            
            # --- AUGMENTATION STACK ---
            aug_imgs = [enhanced]
            aug_imgs.append(cv2.flip(enhanced, 1)) # Flip
            aug_imgs.append(rotate_image(enhanced, -10)) # Rotate Left
            aug_imgs.append(rotate_image(enhanced, 10))  # Rotate Right
            aug_imgs.append(cv2.convertScaleAbs(enhanced, alpha=1, beta=-40)) # Darker
            aug_imgs.append(cv2.convertScaleAbs(enhanced, alpha=1, beta=40))  # Brighter
            
            # Add to datasets
            for face in aug_imgs:
                # 1. For LBPH: Add the image itself
                lbph_faces.append(face)
                lbph_ids.append(user_id)
                
                # 2. For SVM: Extract histogram and add
                hist = desc.describe(face)
                svm_features.append(hist)
                svm_labels.append(user_id)
                
        except Exception as e:
            print(f"Skipping {img_path}: {e}")
            
    return lbph_faces, lbph_ids, svm_features, svm_labels

# --- 4. EXECUTION ---
data_path = r'C:\Users\hp\Desktop\Attendance-System-Using-Face-Recognition\Dataset\training\Cleaned_Training'

faces, ids, features, labels = load_and_augment_data(data_path)

if len(faces) > 0:
    print(f"Total Augmented Samples: {len(faces)}")
    
    # --- TRAIN LBPH ---
    print("Training LBPH Model...")
    lbph = cv2.face.LBPHFaceRecognizer_create(radius=RADIUS, neighbors=NEIGHBORS, grid_x=GRID_X, grid_y=GRID_Y)
    lbph.train(faces, np.array(ids))
    lbph.save('trainer.yml')
    print("Saved 'trainer.yml'")
    
    # --- TRAIN SVM ---
    print("Training SVM Model...")
    svm = SVC(kernel='linear', C=10.0 , gamma='scale', probability=True, random_state=42)
    svm.fit(features, labels)
    joblib.dump(svm, 'svm_face_model.pkl')
    print("Saved 'svm_face_model.pkl'")
    
else:
    print("No data found.")

Processing 139 source images...
Total Augmented Samples: 834

Training LBPH Model...
Saved 'trainer.yml'

Starting SVM Optimization...
Training on 667 samples, Testing on 167 samples...
Fitting 3 folds for each of 30 candidates, totalling 90 fits

Best Settings Found: {'C': 10, 'gamma': 'scale', 'kernel': 'linear'}

--- Accuracy Report ---
Final Test Accuracy: 95.21%

Detailed Report:
              precision    recall  f1-score   support

           1       0.94      0.94      0.94        33
           2       0.92      0.92      0.92        26
           3       0.89      0.94      0.92        18
           4       0.95      0.95      0.95        42
           5       1.00      1.00      1.00        22
           6       1.00      0.96      0.98        26

    accuracy                           0.95       167
   macro avg       0.95      0.95      0.95       167
weighted avg       0.95      0.95      0.95       167

Saved optimized 'svm_face_model.pkl'
